# 01 — Senate Results Inspection and Outcome Layer

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("01_senate_results_inspection", total_steps=11, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook prepares the actual election outcome layer: national vote totals, vote ranks, winner labels, turnout metrics, regional/provincial summaries, and report-ready visuals.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
senate_path = RAW / "senate25-final_updated.csv"
senate = pd.read_csv(senate_path)
print("Senate dataset shape:", senate.shape)
senate.head()


In [ ]:
progress.step("Purpose")
candidate_cols = get_candidate_columns(senate)
candidate_ref = build_candidate_reference(senate)
print("Candidate columns detected:", len(candidate_cols))
display(candidate_ref.head(10))

candidate_ref.to_csv(PROCESSED / "senate_candidate_reference.csv", index=False)


In [ ]:
progress.step("Purpose")
national_results = senate_national_results(senate)
national_results.to_csv(PROCESSED / "senate_national_candidate_results.csv", index=False)
display(national_results.head(20))


In [ ]:
progress.step("Purpose")
# Election operations metrics by region
numeric_cols = ["registeredVoters", "actualVoters", "validBallot", "overVotes", "underVotes", "validVotes", "obtainedVotes"]
for c in numeric_cols:
    senate[c] = pd.to_numeric(senate[c], errors="coerce")

region_turnout = senate.groupby("region", as_index=False)[numeric_cols].sum()
region_turnout["turnout_rate"] = region_turnout["actualVoters"] / region_turnout["registeredVoters"]
region_turnout["undervote_rate"] = region_turnout["underVotes"] / region_turnout["validVotes"]
region_turnout["overvote_rate"] = region_turnout["overVotes"] / region_turnout["validVotes"]
region_turnout = region_turnout.sort_values("turnout_rate", ascending=False)
region_turnout.to_csv(PROCESSED / "region_turnout_metrics.csv", index=False)
region_turnout.head(20)


In [ ]:
progress.step("region_candidate_totals = senate.groupby('region')[candidate_cols].sum().reset_index()")
# Regional candidate rankings
region_candidate_totals = senate.groupby("region")[candidate_cols].sum().reset_index()
region_long = region_candidate_totals.melt(id_vars="region", var_name="candidate_column", value_name="votes")
region_long = region_long.merge(candidate_ref[["candidate_column", "candidate", "party"]], on="candidate_column", how="left")
region_long["regional_vote_rank"] = region_long.groupby("region")["votes"].rank(method="min", ascending=False).astype(int)
region_long.to_csv(PROCESSED / "region_candidate_results_long.csv", index=False)
region_long.sort_values(["region", "regional_vote_rank"]).head(25)


In [ ]:
progress.step("plot_df = national_results.sort_values('total_votes', ascending=True).tail(20)")
# Beautiful visualization: national top candidates
plot_df = national_results.sort_values("total_votes", ascending=True).tail(20)
fig = px.bar(
    plot_df,
    x="total_votes",
    y="candidate",
    orientation="h",
    color="winner_top12",
    title="Top 20 Senate candidates by national vote total",
    labels={"total_votes": "Total votes", "candidate": "Candidate", "winner_top12": "Top 12 winner"},
    text="vote_rank",
)
fig.update_layout(height=760, yaxis_title="", xaxis_title="Total votes", legend_title="Top 12")
fig.update_traces(textposition="outside")
save_plotly(fig, INTERACTIVE / "01_national_top20_vote_ranking.html", FIGURES / "01_national_top20_vote_ranking.png")
fig.show()


In [ ]:
progress.step("plot_df = national_results.sort_values('vote_rank').copy()")
# Winner cutoff chart
plot_df = national_results.sort_values("vote_rank").copy()
plot_df["result_group"] = np.where(plot_df["winner_top12"], "Top 12 winners", "Other candidates")
fig = px.bar(
    plot_df,
    x="vote_rank",
    y="total_votes",
    color="result_group",
    hover_data=["candidate", "party"],
    title="National Senate vote totals with top-12 winner cutoff",
    labels={"vote_rank": "Vote rank", "total_votes": "Total votes", "result_group": "Group"},
)
fig.add_vline(x=12.5, line_dash="dash", annotation_text="Top 12 cutoff", annotation_position="top right")
fig.update_layout(height=560)
save_plotly(fig, INTERACTIVE / "01_winner_cutoff_vote_chart.html", FIGURES / "01_winner_cutoff_vote_chart.png")
fig.show()


In [ ]:
progress.step("fig = px.bar(")
fig = px.bar(
    region_turnout.sort_values("turnout_rate"),
    x="turnout_rate",
    y="region",
    orientation="h",
    title="Turnout rate by region",
    labels={"turnout_rate": "Turnout rate", "region": "Region"},
)
fig.update_layout(height=680, xaxis_tickformat=".0%")
save_plotly(fig, INTERACTIVE / "01_turnout_rate_by_region.html", FIGURES / "01_turnout_rate_by_region.png")
fig.show()


In [ ]:
progress.step("regional_top = region_long[region_long['regional_vote_rank'] <= 5].sort_values(['region', 'regional_vote_rank'])")
# Top candidates by region, useful for Philippine geographic context
regional_top = region_long[region_long["regional_vote_rank"] <= 5].sort_values(["region", "regional_vote_rank"])
regional_top.to_csv(TABLES / "regional_top5_candidates.csv", index=False)
regional_top.head(30)


In [ ]:
progress.step("national_top12 = national_results.query('winner_top12')['candidate'].tolist()")
# Heatmap: regional rank of national top 12
national_top12 = national_results.query("winner_top12")["candidate"].tolist()
heat_df = region_long[region_long["candidate"].isin(national_top12)].pivot(index="region", columns="candidate", values="regional_vote_rank")
plt.figure(figsize=(18, 8))
sns.heatmap(heat_df, annot=True, fmt=".0f", cmap="viridis_r", cbar_kws={"label": "Regional rank; lower is better"})
plt.title("Regional rank heatmap for national top 12 Senate winners")
plt.xlabel("Candidate")
plt.ylabel("Region")
plt.tight_layout()
plt.savefig(FIGURES / "01_regional_rank_heatmap_top12.png", dpi=220, bbox_inches="tight")
plt.show()


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
